# `Reporting Excel automatisé en Python`
*Oscar JOSEPH--GENESLAY*
**Data :** SuperStore Dataset
Source : *[Kaggle - SuperStore Dataset par vivek468](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final?resource=download)*

# Importation des packages

In [182]:
import pandas as pd
from datetime import datetime
import openpyxl
from openpyxl.chart import BarChart, Reference, DoughnutChart, series
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import FORMULAE
from openpyxl.chart.series import DataPoint
from openpyxl.chart.label import DataLabelList
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.worksheet.datavalidation import DataValidation

# Importation des données

In [183]:
dataset = pd.read_csv(
    "https://minio.lab.sspcloud.fr/oscar04/Superstore/Superstore.csv",
    encoding="windows-1252"
)

dataset.head(3)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [184]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

# Reformatage des colonnes

In [185]:
# Retypage
dataset["Row ID"] = str(dataset["Row ID"])
dataset["Postal Code"] = str(dataset["Postal Code"])


# Transformation en format date
dataset["Order Date"] = pd.to_datetime(dataset["Order Date"], format='%d/%m/%Y', errors='coerce')
dataset["Ship Date"] = pd.to_datetime(dataset["Ship Date"], format='%d/%m/%Y', errors='coerce')

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   object        
 1   Order ID       9994 non-null   object        
 2   Order Date     4042 non-null   datetime64[ns]
 3   Ship Date      3898 non-null   datetime64[ns]
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City           9994 non-null   object        
 10  State          9994 non-null   object        
 11  Postal Code    9994 non-null   object        
 12  Region         9994 non-null   object        
 13  Product ID     9994 non-null   object        
 14  Category       9994 non-null   object        
 15  Sub-Category   9994 n

# Fichier du reporting

## Création du fichier Excel

In [186]:
path = "../output/reporting.xlsx"
with pd.ExcelWriter(path, engine="openpyxl") as writer:
    dataset.to_excel(writer, sheet_name="DATA", index=False)

print("✅ reporting.xlsx créé")

✅ reporting.xlsx créé


## Chargement du reporting en mémoire

In [187]:
# Charger le classeur (workbook) en mémoire
wb = openpyxl.load_workbook('../output/reporting.xlsx')

# Création d'une feuille visualisations
ws_visualisations = wb.create_sheet(title="VISUALISATIONS")

# Définir la feuille DATA en variable python
ws_data = wb["DATA"]

# wb est un objet Python — on ne peut pas afficher son contenu avec un simple print
print(type(wb))
print('Feuilles disponibles :', wb.sheetnames)

<class 'openpyxl.workbook.workbook.Workbook'>
Feuilles disponibles : ['DATA', 'VISUALISATIONS']


# Création des KPIs

### KPIs numériques

##### CA ; Profits ; Quantité

In [188]:
# Création fonction d'automatisation
def numerical_kpis(cell, colonne, cell_format):
    c = ws_visualisations[cell]
    c.value = f"=SUMIF(DATA!M:M, VISUALISATIONS!D1, DATA!{colonne}:{colonne})"
    c.number_format = cell_format
    c.font = Font(name="Calibri", size=16, bold=True, color="1F497D")
    c.alignment = Alignment(horizontal="left", vertical="center")

    # Ajustement automatique de la largeur pour éviter les ####
    col_letter = c.column_letter
    # On récupère la largeur actuelle ou une valeur par défaut de 10
    current_width = ws_visualisations.column_dimensions[col_letter].width or 10
    # définit une largeur de marge
    needed_width = max(len(cell_format) + 5, 18)
    # applique la largeur
    ws_visualisations.column_dimensions[col_letter].width = max(current_width, needed_width)
    ws_visualisations.row_dimensions[c.row].height = 30

In [189]:
# CA
numerical_kpis("B4", "R", "#,##0.0 $")

# Profit
numerical_kpis("F4", "U", "#,##0.0 $")

# Quantité
numerical_kpis("J4", "S", "#,##0")

##### Temps de livraison

La fonction de calcul de différence entre date existe dans openpyxl

In [190]:
"DATEDIF" in FORMULAE

True

Application de la formule datedif sur chaque ligne où c'est calculable

In [191]:
for row_cell in range(2, len(dataset)+2):
    cell_application = f"V{row_cell}"
    date_debut = f"C{row_cell}"
    date_fin = f"D{row_cell}"
    ws_data[cell_application] = f'=IF(OR({date_debut} = "", {date_fin} = ""), "", DATEDIF({date_debut}, {date_fin}, "D"))'

Création du KPI moyenne 

In [192]:
"AVERAGEIF" in FORMULAE

True

In [193]:
ws_visualisations["O4"] = "=ROUND(AVERAGEIF(DATA!M:M, VISUALISATIONS!D1, DATA!V:V),1)"

In [194]:
c = ws_visualisations["O4"]
cell_format = c.number_format or ""
c.font = Font(name="Calibri", size=16, bold=True, color="1F497D")
c.alignment = Alignment(horizontal="left", vertical="center")

# Ajustement de la hauteur de la ligne 4
ws_visualisations.row_dimensions[4].height = 30
# ajustement de la largeur de la colonne O
col_letter = "O"
current_width = ws_visualisations.column_dimensions[col_letter].width or 10
# Définit une largeur de marge basée sur le format
needed_width = max(len(cell_format) + 5, 18)
# Appliquer la largeur finale
ws_visualisations.column_dimensions[col_letter].width = max(current_width, needed_width)

### TOP10 des sous-catégories avec le plus de profits

1. Calcul des profits par sous catégories et création du top 10

La fonction UNIQUE n'existe pas dans openpyxl

In [195]:
"UNIQUE" in FORMULAE

False

Calcul du nombre de sous catégories

In [196]:
len_sous_cat = len(dataset["Sub-Category"].unique())+1
# Calcul du nombre de sous catégories
len_dict ={}
for col in ["Sub-Category"]:
    len_dict["len_sous_cat"] = len(dataset[col].unique())+1 # on rajoute +1 car python commence à 0
len_dict

{'len_sous_cat': 18}

Application de la formule UNIQUE

In [197]:
formula = "=_xlfn.UNIQUE(DATA!P:P)"
ws_data['Z2'] = ArrayFormula(f"Z2:Z{len_dict['len_sous_cat']+1}", formula)

Calculer le profits par sous catégories

In [198]:
for num_cell in range(3, len_dict['len_sous_cat']+2):
    cell = "AA" + str(num_cell)
    ws_data[cell] = "=SUMIFS(DATA!U:U,DATA!P:P,DATA!Z" + str(num_cell) + ",DATA!M:M,Visualisations!$D$1)"

La fonction "SORT" n'existe pas dans openpyxl

In [199]:
"SORT" in FORMULAE

False

Application du tri du tableau de données

In [200]:
# Trier le tableau de données
formula = "=_xlfn.SORT(DATA!Z3:AA19,2,-1)"
ws_data['AC3'] = ArrayFormula("AC3:AD19", formula)

2. Création du graphique à barres

In [201]:
# Création du graphique à barres
chart_top10 = BarChart()
chart_top10.type = "col"
chart_top10.title = "TOP 10 du profit en fonction des sous-catégories"
chart_top10.width = 18
chart_top10.height = 10

# Récupération des données de profits
data_ref = Reference(ws_data, min_col=30, min_row=3, max_row=12)

# Récupération des labels
cats_ref = Reference(ws_data, min_col=29, min_row=4, max_row=12)

# Appliquer les paramètres au graphique
chart_top10.add_data(data_ref, titles_from_data=True)
chart_top10.set_categories(cats_ref)

# Ajouter les étiquettes de données
chart_top10.dataLabels = DataLabelList()
chart_top10.dataLabels.showVal = True
chart_top10.dataLabels.numFmt = '#,##0'

# Titres des axes
chart_top10.y_axis.title = 'Profits (en $)'
chart_top10.x_axis.title = 'Sous-catégories'

# Légende
chart_top10.legend = None

# Placement du graphique
ws_visualisations.add_chart(chart_top10, "B7")

### Pourcentage de CA par catégories

Écrire les valeurs uniques des catégories disponibles

In [202]:
len_cat = len(dataset["Category"].unique())+1
# Calcul du nombre de sous catégories
len_dict ={}
for col in ["Category"]:
    len_dict["len_cat"] = len(dataset[col].unique())+1 # on rajoute +1 car python commence à 0
formula = "=_xlfn.UNIQUE(DATA!O:O)"
ws_data['Z25'] = ArrayFormula(f"Z25:Z{len_dict['len_cat']+24}", formula)

Calculer le CA pour chaque catégorie

In [203]:
for row_cell in range(26, 29):
    ws_data[f"AA{row_cell}"] = f"=SUMIFS(DATA!R:R, DATA!O:O, DATA!Z{row_cell}, DATA!M:M, Visualisations!$D$1)"

Calculer le pourcentage du CA par catégorie

In [204]:
ws_data["AA29"] = "=SUM(AA26:AA28)"
for row_cell in range(26, 29):
    ws_data[f"AB{row_cell}"] = f"=AA{row_cell} / AA29"
    ws_data[f"AB{row_cell}"].number_format = '0.00%'

Génération du graphique

In [205]:
chart_percent = DoughnutChart()
chart_percent.width = 18
chart_percent.height = 10
labels = Reference(ws_data, min_col=26, min_row=26, max_row=28)
data = Reference(ws_data, min_col=28, min_row=25, max_row=28)
chart_percent.add_data(data, titles_from_data=True)
chart_percent.set_categories(labels)
chart_percent.title = "Part du CA pour chaque catégorie"

chart_percent.dataLabels = DataLabelList()
chart_percent.dataLabels.showPercent = True   # Afficher les pourcentages

# Cut the first slice out of the doughnut
slices = [DataPoint(idx=i) for i in range(4)]
plain, jam, lime, chocolate = slices
chart_percent.series[0].data_points = slices
plain.graphicalProperties.solidFill = "A9E6EB"
jam.graphicalProperties.solidFill = "B7A9EB"
lime.graphicalProperties.solidFill = "EBA9D7"

ws_visualisations.add_chart(chart_percent, "B27")


# Création de l'aspect design

### Le titre

In [206]:
ws_visualisations["I1"] = "Performances économique"
ws_visualisations["I1"].font = Font(bold=True, size=28, color="185EB8")

### KPIs numériques

In [207]:
def design_numerical_kpis(cell, nom):
    ws_visualisations[cell] = nom
    ws_visualisations[cell].font = Font(bold=True, size=16)

In [208]:
# CA
design_numerical_kpis("B3", "Chiffre d'affaires")

# Profits
design_numerical_kpis("F3", "Profits")

# Quantités
design_numerical_kpis("J3", "Volume des ventes")

# Temps livraison
design_numerical_kpis("O3", "Temps de livraison moyen (en jours)")


# Ajout du filtre

In [209]:
liste_regions = ",".join(dataset["Region"].unique())
liste_regions

'South,West,Central,East'

In [210]:
ws_visualisations["C1"] = "Choisir Région :"
ws_visualisations["D1"] = dataset["Region"][0]  # Valeur par défaut

# Création de la règle de validation (liste de valeurs)
dv = DataValidation(
    type="list", formula1=f'"{liste_regions}"', allow_blank=True
)
dv.error = "Veuillez choisir une région de la liste"
dv.errorTitle = "Saisie invalide"

# Ajouter la règle à la feuille et l'appliquer à la cellule D1
ws_visualisations.add_data_validation(dv)
dv.add(ws_visualisations["D1"])

# Export du reporting

In [211]:
wb.save("../output/reporting.xlsx")
print(f"✅ Fichier Excel exporté vers {path}")

✅ Fichier Excel exporté vers ../output/reporting.xlsx
